In [1]:
import torch

from tts.config.ndaligner.training_module_config import NDAlignerTrainingModuleConfigs
from tts.config.utils.io import load_config
from tts.models.ndaligner import init_nd_aligner_training_module

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## INIT Path and Configs

In [22]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
aligner_training_module_cfg_path = "./runs/nd_aligner_vctk_10ms_coupling_dec_20260711-154829/model_config.json"
aligner_training_module_ckpt_path = "./runs/nd_aligner_vctk_10ms_coupling_dec_20260711-154829/checkpoints_timit_bae/best_step_timit_bae_0.019927_step_266500_epoch_39.pth"


## INIT Models

In [23]:
model_config = load_config(aligner_training_module_cfg_path, NDAlignerTrainingModuleConfigs,)

aligner_training_module = init_nd_aligner_training_module(
    config=model_config,
    device=device,
)
aligner_training_module.load_checkpoint(
    ckpt_path=aligner_training_module_ckpt_path,
    device=device,
)
aligner = aligner_training_module.nd_aligner.eval()

SpeechBrain ECAPA-TDNN SpeakerEncoder loaded on cuda.
Loading nested state_dict from key 'model' in ./runs/nd_aligner_vctk_10ms_coupling_dec_20260711-154829/checkpoints_timit_bae/best_step_timit_bae_0.019927_step_266500_epoch_39.pth
⚠️ Checkpoint load summary (strict=False):
  - Unexpected top-level modules: ['vocoder']
Checkpoint loading process finished.


## INIT BenchMarkers (TIMIT)

In [24]:
from tts.benchmark.timit.benchmarker import TIMITBenchMarker
from tts.models.utils.input_maker import AlignerInputMaker

TIMIT_ROOT = "/shared/data_zfs/blue2959/TIMIT/TEST"

input_maker = AlignerInputMaker(
    audio_config=model_config.nd_aligner.audio,
    preprocess_config=model_config.nd_aligner.preprocess,
    tokenizer_type=model_config.nd_aligner.tokenizer_type,
    zero_nonspeech_region=True,
    trim_nonspeech_region=True,
    suppress_impulsive_peak=False,
    reduce_noise=False,
    device="cuda"
)

timit_benchmarker = TIMITBenchMarker(
    root_dir=TIMIT_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=22_050,
    hyp_hop_length=256,
    input_maker=input_maker,
    hyp_ignore_symbols=input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

SpeechBrain ECAPA-TDNN SpeakerEncoder loaded on cuda.
[TIMITBenchMarker] Found 1680 valid (WRD, WAV, TXT) triplets.


In [ ]:
with torch.no_grad():
    metrics = timit_benchmarker.__call__(
        aligner=aligner,
        vocoder=None,
        max_test_samples=None,
    )

print(metrics.word_boundary_error)
print(metrics.p_word_100ms)
print(metrics.p_word_50ms)
print(metrics.p_word_25ms)
print(metrics.p_word_10ms)

Computing Alignments: 100%|██████████| 1680/1680 [02:24<00:00, 11.61it/s]

0.020024007186293602
98.6225962638855
92.78354048728943
75.03504753112793
39.36632573604584


: 

## INIT BenchMarkers (Buckeye)

In [20]:
from tts.benchmark.timit.benchmarker import TIMITBenchMarker
from tts.models.utils.input_maker import AlignerInputMaker


assert aligner.input_maker is not None

input_maker = AlignerInputMaker(
    audio_config=model_config.nd_aligner.audio,
    preprocess_config=model_config.nd_aligner.preprocess,
    tokenizer_type=model_config.nd_aligner.tokenizer_type,
    zero_nonspeech_region=True,
    trim_nonspeech_region=True,
    suppress_impulsive_peak=False,
    reduce_noise=True,
    device="cuda"
)

BUCKEYE_ROOT = "/shared/data_zfs/blue2959/Buckeye-grid" # (compatible with timit benchmarker!)

buckeye_benchmarker = TIMITBenchMarker(
    root_dir=BUCKEYE_ROOT,
    ref_audio_sr=16_000,
    hyp_audio_sr=22_050,
    hyp_hop_length=256,
    input_maker=input_maker,
    hyp_ignore_symbols=input_maker.tokenizer.ignore_symbols,
    max_ref_words_per_hyp_word=5,
)

SpeechBrain ECAPA-TDNN SpeakerEncoder loaded on cuda.
[TIMITBenchMarker] Found 19273 valid (WRD, WAV, TXT) triplets.


In [21]:
with torch.inference_mode():
    metrics = buckeye_benchmarker.__call__(
        aligner=aligner,
        vocoder=None,
        max_test_samples=500,
    )

print(metrics.word_boundary_error)
print(metrics.p_word_100ms)
print(metrics.p_word_50ms)
print(metrics.p_word_25ms)
print(metrics.p_word_10ms)

Computing Alignments: 100%|██████████| 500/500 [01:01<00:00,  8.13it/s]

0.032187893986701965
92.7675724029541
87.0098888874054
68.92440915107727
35.03178954124451
